# Fraud Detection Project

In this project, we will:
1. Load and explore the dataset
2. Visualize patterns in the data
3. Train a machine learning model to detect fraud
4. Evaluate the model properly
5. Save the model for use in the app

Dataset: [Fraud Detection Dataset on Kaggle](https://www.kaggle.com/datasets/amanalisiddiqui/fraud-detection-dataset)

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')

print('Libraries loaded successfully')

## Step 2: Load the Dataset

Make sure the CSV file is in the same folder as this notebook.

In [ ]:
df = pd.read_csv('AIML Dataset.csv')

print('Dataset shape:', df.shape)
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

df.head()

## Step 3: Basic Data Exploration

Let us understand what columns we have and check for missing values.

In [ ]:
# Check column types and non-null counts
df.info()

In [ ]:
# Check for missing values
print('Missing values in each column:')
print(df.isnull().sum())

In [ ]:
# How many fraud vs normal transactions?
print('Fraud vs Normal transaction count:')
print(df['isFraud'].value_counts())
print()
print('As percentage:')
print(df['isFraud'].value_counts(normalize=True) * 100)

## Step 4: Visualizations

We will create simple charts to understand the data better.

In [ ]:
# Bar chart: Fraud vs Normal
df['isFraud'].value_counts().plot(
    kind='bar',
    color=['steelblue', 'salmon'],
    title='Fraud vs Normal Transactions'
)
plt.xlabel('0 = Normal, 1 = Fraud')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart: Transaction types
df['type'].value_counts().plot(
    kind='bar',
    color='skyblue',
    title='Transaction Types'
)
plt.xlabel('Transaction Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Which transaction types have the most fraud?
fraud_by_type = df.groupby('type')['isFraud'].mean().sort_values(ascending=False)

fraud_by_type.plot(
    kind='bar',
    color='salmon',
    title='Fraud Rate by Transaction Type'
)
plt.xlabel('Transaction Type')
plt.ylabel('Fraud Rate (0 to 1)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Transaction amount distribution (log scale because amounts vary a lot)
sns.histplot(
    np.log1p(df['amount']),
    bins=80,
    kde=True,
    color='green'
)
plt.title('Transaction Amount Distribution (log scale)')
plt.xlabel('log(Amount + 1)')
plt.tight_layout()
plt.show()

## Step 5: Feature Engineering

We will create two new features from existing balance columns.
The idea: if money leaves the origin account, `balanceDiffOrig` will be positive.
If money arrives at the destination, `balanceDiffDest` will be positive.
These differences are often very different in fraud transactions.

In [ ]:
# New feature: how much did the origin balance decrease?
df['balanceDiffOrig'] = df['oldbalanceOrg'] - df['newbalanceOrig']

# New feature: how much did the destination balance increase?
df['balanceDiffDest'] = df['newbalanceDest'] - df['oldbalanceDest']

print('New features added:')
df[['balanceDiffOrig', 'balanceDiffDest']].head()

## Step 6: Prepare Data for Model

We drop columns that are not useful for prediction:
- `nameOrig` and `nameDest` are just account IDs (like names, not useful patterns)
- `isFlaggedFraud` is a rule-based flag from the bank system, not a real feature we should use

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Drop columns we do not need
df_model = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'])

# Separate features (X) and target (y)
X = df_model.drop(columns=['isFraud'])
y = df_model['isFraud']

print('Features (X) shape:', X.shape)
print('Target (y) shape:', y.shape)
print()
print('Feature columns:', list(X.columns))

In [ ]:
# Split into train and test sets
# stratify=y makes sure both sets have the same fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print('Training set size:', X_train.shape[0])
print('Test set size:', X_test.shape[0])

## Step 7: Build and Train the Model

We use a Random Forest classifier. It works better than Logistic Regression for fraud detection.

We use `n_estimators=100` (100 decision trees). We keep it small so it does not freeze your device.

We also use `class_weight='balanced'` because there are very few fraud cases compared to normal ones.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Define which columns are numeric and which are categorical
numeric_cols = [
    'step', 'amount',
    'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest',
    'balanceDiffOrig', 'balanceDiffDest'
]
categorical_cols = ['type']

# Preprocessor: scale numbers, encode categories
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(drop='first'), categorical_cols)
])

# Full pipeline: preprocess then classify
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1           # use all CPU cores to speed up training
    ))
])

print('Training the model... this may take a minute')
pipeline.fit(X_train, y_train)
print('Training complete')

## Step 8: Evaluate the Model

For fraud detection, accuracy alone is misleading.
If 99% of transactions are normal, a model that always says "normal" gets 99% accuracy but catches zero fraud.

We care about:
- **Recall (fraud class)**: out of all real fraud cases, how many did we catch?
- **Precision (fraud class)**: out of all cases we flagged as fraud, how many were actually fraud?
- **ROC-AUC**: overall model quality (1.0 is perfect, 0.5 is random)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

print('ROC-AUC Score:', round(roc_auc_score(y_test, y_proba), 4))

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Normal', 'Fraud'],
    yticklabels=['Normal', 'Fraud']
)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print('True Negatives  (Normal predicted Normal):', cm[0][0])
print('False Positives (Normal predicted Fraud) :', cm[0][1])
print('False Negatives (Fraud predicted Normal) :', cm[1][0])
print('True Positives  (Fraud predicted Fraud)  :', cm[1][1])

In [ ]:
# Which features are most important to the model?
feature_names = (
    numeric_cols +
    list(pipeline.named_steps['preprocessor']
         .named_transformers_['cat']
         .get_feature_names_out(categorical_cols))
)

importances = pipeline.named_steps['classifier'].feature_importances_

feat_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=feat_df, x='importance', y='feature', color='steelblue')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

## Step 9: Save the Model

We save the trained pipeline so the Streamlit app can use it without retraining.

In [ ]:
import joblib

joblib.dump(pipeline, 'Fraud_detection_pipeline.pkl')

print('Model saved as Fraud_detection_pipeline.pkl')

## Summary

What we did in this project:

1. Loaded the fraud detection dataset
2. Explored the data (shapes, nulls, fraud rate)
3. Visualized transaction types and fraud patterns
4. Engineered two new features: `balanceDiffOrig` and `balanceDiffDest`
5. Built a preprocessing pipeline with scaling and encoding
6. Trained a Random Forest classifier with balanced class weights
7. Evaluated using classification report, confusion matrix, and ROC-AUC
8. Saved the model for deployment in the Streamlit app

Key finding: Fraud transactions mostly occur in TRANSFER and CASH_OUT types, and the balance difference features are strong predictors.